# 00 - Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/PyADM1ODE_calibration/blob/main/notebooks/00_quickstart.ipynb)

The whole calibration workflow in one pass: measurements in, calibrated parameter out,
model checked against the data.

> **This is the shortcut.** Notebooks 01 to 06 walk through the
> same steps one at a time and explain *why* each one is done. 

In [ ]:
# On Colab, run this once; locally it does nothing.
# !git clone -q https://github.com/dgaida/PyADM1ODE_calibration.git
# %pip install -q ./PyADM1ODE_calibration
# %cd PyADM1ODE_calibration/notebooks

## 1. Measurements

`make_twin_measurements` simulates the demo plant with a known parameter value and adds
noise. That is the point of a *twin* dataset: the calibration can be judged against the
truth instead of against a plausible-looking curve.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from demo_plant import (
    DEFAULT_K_HYD_CH,
    TRUE_PARAMETERS,
    build_demo_plant,
    make_twin_measurements,
    simulate,
)

from pyadm1ode_calibration import InitialCalibrator

measurements = make_twin_measurements(days=5, noise=0.02, seed=0)
print(f"{len(measurements.data)} hourly rows, channels: "
      f"{[c for c in measurements.data.columns if not c.startswith('Q_sub')]}")
print(f"true k_hyd_ch (hidden from the calibrator): {TRUE_PARAMETERS['k_hyd_ch']}")
print(f"model default the calibration starts from : {DEFAULT_K_HYD_CH}")

## 2. Calibrate

One parameter, one objective channel, Nelder-Mead. `sensitivity_analysis=False` keeps it
to a handful of simulations, which is all this well-posed problem needs.

In [ ]:
result = InitialCalibrator(build_demo_plant(days=5), verbose=False).calibrate(
    measurements,
    parameters=["k_hyd_ch"],
    bounds={"k_hyd_ch": (1.0, 15.0)},
    objectives=["Q_gas"],
    method="nelder_mead",
    max_iterations=20,
    sensitivity_analysis=False,
)

fitted = result.parameters["k_hyd_ch"]
true = TRUE_PARAMETERS["k_hyd_ch"]
print(f"success         : {result.success}")
print(f"k_hyd_ch found  : {fitted:.3f}   (true {true:.3f}, off by {fitted / true - 1:+.1%})")
print(f"objective       : {result.objective_value:.4f}   (normalised RMSE)")
print(f"simulations run : {len(result.history)}")

## 3. Did the model get better?

The parameter being close to the truth is reassuring, but the question that matters is
whether the *simulation* now follows the measurements. Below: the measured `Q_gas` against
the model before and after calibration.

In [ ]:
plant = build_demo_plant(days=5)
t = measurements.data.index
measured = measurements.data["Q_gas"].to_numpy()
before = np.asarray(simulate(plant, measurements, {"k_hyd_ch": DEFAULT_K_HYD_CH})["Q_gas"], float)
after = np.asarray(simulate(plant, measurements, {"k_hyd_ch": fitted})["Q_gas"], float)

rmse = lambda y: float(np.sqrt(np.mean((y - measured) ** 2)))
print(f"RMSE(Q_gas) before: {rmse(before):7.1f} m3/d")
print(f"RMSE(Q_gas) after : {rmse(after):7.1f} m3/d")

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11, 3.4))
ax.plot(t, measured, color="0.45", lw=0.9, alpha=0.8, label="measured")
ax.plot(t, before, ls="--", color="tab:red", label=f"before (k_hyd_ch={DEFAULT_K_HYD_CH})")
ax.plot(t, after, color="tab:blue", label=f"after (k_hyd_ch={fitted:.2f})")
# The first hours are the start-up transient; scale the view to the plateau, where the
# parameter is actually visible. The fit itself uses the whole window.
plateau = measured[12:]
ax.set_ylim(plateau.min() * 0.97, plateau.max() * 1.06)
ax.set_xlim(t[0], t[-1])
ax.set_ylabel("Q_gas [m3/d]")
ax.set_title("Measurement against model", fontsize=10)
ax.tick_params(axis="x", rotation=30, labelsize=7)
ax.legend(fontsize=7, loc="lower right")

# Plot the running best, not the raw sequence: a candidate the simulator rejects gets a
# huge penalty value, and a single one of those would squash the real descent flat.
obj = np.array([h["objective"] for h in result.history], float)
penalised = int((obj > 1e6).sum())
ax2.plot(np.minimum.accumulate(obj), marker="o", ms=3)
ax2.set_xlabel("simulation")
ax2.set_ylabel("best objective so far")
ax2.set_title(f"What the optimiser did  ({penalised} rejected)", fontsize=10)

fig.tight_layout()

## Where to go from here

| Question | Notebook |
| --- | --- |
| My data has outliers and gaps, what now? | `01_explore_measurements` |
| How far off is an uncalibrated model? | `02_model_vs_measurement` |
| What do the calibration options mean? | `03_first_calibration` |
| Which parameters are even worth fitting? | `04_which_parameters_matter` |
| Did I fit the signal or the noise? | `05_train_test_and_residuals` |
| The plant drifts, how do I keep up? | `06_online_recalibration` |